In [1]:
import keras
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import sklearn as sk

In [2]:
DATA_DIR = "mu3e_trigger_data"
SIGNAL_DATA_FILE = f"{DATA_DIR}/run42_sig_positions.npy"
BACKGROUND_DATA_FILE = f"{DATA_DIR}/run42_bg_positions.npy"

max_barrel_radius = 86.3
max_endcap_distance = 372.6

In [3]:
signal_data = np.load(SIGNAL_DATA_FILE)
background_data = np.load(BACKGROUND_DATA_FILE)

In [4]:
background_data = background_data[~(background_data[:, :, 0] == -1).all(axis=1)]
signal_data = signal_data[~(signal_data[:, :, 0] == -1).all(axis=1)]

In [5]:
def RotMatrix3D(angle):
    return np.array(
        [
            [np.cos(angle), -np.sin(angle), 0],
            [np.sin(angle), np.cos(angle), 0],
            [0, 0, 1],
        ]
    )

def generate_contrastive_data(data, num_angles = 4):
    angles = np.linspace(0, 2 * np.pi, num_angles, endpoint=False)
    mask = (data == -1).all(axis = -1)
    rotation_matrices = np.stack([RotMatrix3D(angle) for angle in angles])
    rotated_data = np.einsum("ijk,lkm->ijlm", data, rotation_matrices)
    rotated_data[mask, :] = -1
    return [rotated_data[:, :, i, :] for i in range(num_angles)]

In [15]:
from src.model.components import (
    GenerateMask,
    PointTransformerBlock,
    SelfAttentionStack,
    PoolingAttentionBlock,
    MLP,
)

input_layer = keras.layers.Input(shape=(256, 3))

mask = GenerateMask(padding_value=-1)(input_layer)

input_embedding = PointTransformerBlock(
    hidden_dim=16,
    name="point_transformer_block"
)(input_layer, mask)


transformer = SelfAttentionStack(stack_size=3, num_heads=8, key_dim=16)(
    input_embedding, mask
)

pooling = keras.layers.GlobalAveragePooling1D()(transformer, mask = mask)

latent_output = MLP(
    output_dim=16,
    activation="linear",
    name="final_latent_output"
)(pooling)

embedding_model = keras.Model(inputs=input_layer, outputs=latent_output, name="embedding_model")

In [16]:
def build_siamese_network(input_shape, base_model, num_contrastive_views=4):
    # Create multiple input layers
    views = [keras.layers.Input(shape=input_shape, name=f"input_view_{i}") for i in range(num_contrastive_views)]
    
    # Apply shared model to each input
    model_outputs = [base_model(view) for view in views]
        
    # Concatenate embeddings
    concatenated_views = keras.layers.Concatenate(axis=-1, name="concatenated_embeddings")(model_outputs)
    
    # Build Model
    return keras.Model(inputs=views, outputs=concatenated_views, name="siamese_network")


In [17]:
class CovarianceLoss(keras.losses.Loss):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def call(self, _, y_pred):
        z = y_pred  # shape: (batch_size, latent_dim)
        batch_size = tf.cast(tf.shape(z)[0], tf.float32)
        z_size = tf.cast(tf.shape(z)[1], tf.float32)

        # Compute covariance matrix
        z_centered = z - tf.reduce_mean(z, axis=0)
        cov_matrix = tf.matmul(z_centered, z_centered, transpose_a=True) / (
            batch_size - 1.0
        )

        off_diag_mask = 1.0 - tf.eye(z_size)
        loss = tf.reduce_sum(tf.square(cov_matrix * off_diag_mask)) / z_size
        return loss

class VarianceLoss(keras.losses.Loss):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def call(self, _, y_pred):
        z = y_pred  # shape: (batch_size, latent_dim)
        return tf.reduce_mean(tf.maximum(0., 1- tf.sqrt(tf.math.reduce_variance(z, axis=0) + 1e-4)))

class SWDLoss(tf.keras.losses.Loss):
    def __init__(self, reg_weight=1.0, num_projections=50, name="swd_loss"):
        super().__init__(name=name)
        self.reg_weight = reg_weight
        self.num_projections = num_projections

    def call(self, y_true, y_pred):
        z = y_pred

        prior_z = tf.random.normal(tf.shape(z))
        batch_size = tf.shape(z)[0]
        latent_dim = tf.shape(z)[1]

        # Random directions
        projections = tf.random.normal([self.num_projections, latent_dim])
        projections = tf.math.l2_normalize(projections, axis=-1)  # [P, D]

        proj_z = tf.linalg.matmul(z, projections, transpose_b=True)  # [B, P]
        proj_prior = tf.linalg.matmul(prior_z, projections, transpose_b=True)  # [B, P]

        proj_z_sorted = tf.sort(proj_z, axis=0)
        proj_prior_sorted = tf.sort(proj_prior, axis=0)

        swd = tf.reduce_mean(tf.square(proj_z_sorted - proj_prior_sorted))

        return self.reg_weight * swd
    
class ContrastiveLoss(keras.losses.Loss):
    def __init__(self, contrastive_views = 4, var_mtpl = 1, cov_mtpl = 1, **kwargs):
        super().__init__(**kwargs)
        self.contrastive_views = contrastive_views
        self.var_mtpl = var_mtpl
        self.cov_mtpl = cov_mtpl
        self.cov_loss = CovarianceLoss()
        self.var_loss = VarianceLoss()
        self.swd_loss = SWDLoss()
    
    def call(self, y_true, y_pred):
        outputs = tf.split(y_pred, self.contrastive_views, axis=-1)
        loss = 0.0
        for i in range(self.contrastive_views):
            # Compute pairwise contrastive loss
            for j in range(i + 1, self.contrastive_views):
                loss += tf.reduce_mean(
                    tf.square(outputs[i] - outputs[j]) 
                ) / (self.contrastive_views - 1)
            loss += self.var_mtpl * self.var_loss(outputs[i], outputs[i])
            loss += self.cov_mtpl * self.cov_loss(outputs[i], outputs[i])
            loss += self.swd_loss(outputs[i], outputs[i])
        return loss

In [18]:
from sklearn.model_selection import train_test_split
train_bg_data, test_bg_data = train_test_split(
    background_data, test_size=0.2, random_state=42, shuffle=True
)

In [19]:
contrast_views = 3
contrasted_train_bg_data = generate_contrastive_data(train_bg_data[:10000], contrast_views)
siamese_model = build_siamese_network(
    input_shape=(256, 3),
    base_model=embedding_model,
    num_contrastive_views=contrast_views,
)
siamese_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=ContrastiveLoss(contrastive_views=contrast_views, var_mtpl=1, cov_mtpl=1),
)
siamese_model.summary()

Model: "siamese_network"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_view_0        │ (None, 256, 3)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_view_1        │ (None, 256, 3)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_view_2        │ (None, 256, 3)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_model     │ (None, 16)        │     27,600 │ input_view_0[0][… │
│ (Functional)        │                   │            │ input_view_1[0][… │
│                     │                   │            │ input_view_2[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenated_embed… │ (None, 48)        │          0 │ embedding_model[… │
│ (Concatenate)       │                   │            │ embedding_model[… │
│                     │                   │            │ embedding_model[… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 27,600 (107.81 KB)

 Trainable params: 27,600 (107.81 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
siamese_model.fit(
    contrasted_train_bg_data,
    np.zeros((contrasted_train_bg_data[0].shape[0], 16 * contrast_views)),  # Dummy labels
    batch_size=512,
    epochs=10,
    validation_split=0.2,
)

Epoch 1/10
 1/16 ━━━━━━━━━━━━━━━━━━━━ 40:38 163s/step - loss: 5.7138

In [ ]:
embedded_bg_train_data = embedding_model.predict(train_bg_data)


In [ ]:
from src.model import AutoEncoder

autoencoder = AutoEncoder(
    input_size=16,
    latent_dim=4,
    nodes=4,
    name="autoencoder"
)
autoencoder.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.MeanSquaredError(),
)

In [ ]:
autoencoder.fit(
    embedded_bg_train_data,
    embedded_bg_train_data,
    epochs=1000,
    batch_size=32,
    validation_split=0.2,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=50, restore_best_weights=True
        )
    ],
)

In [ ]:
embedded_bg_test_data = embedding_model.predict(test_bg_data[:10000])
embedded_signal_data = embedding_model.predict(signal_data[:10000])

In [ ]:
diff_bg_test = np.linalg.norm(
    embedded_bg_test_data - autoencoder.predict(embedded_bg_test_data), axis=1
)
diff_signal = np.linalg.norm(
    embedded_signal_data - autoencoder.predict(embedded_signal_data), axis=1
)
diff_bg_train = np.linalg.norm(
    embedded_bg_train_data - autoencoder.predict(embedded_bg_train_data), axis=1
)

In [ ]:
autoencoder.evaluate(
    embedded_bg_train_data,
    embedded_bg_train_data,
    batch_size=32,
)

In [ ]:
autoencoder.evaluate(
    embedded_bg_test_data,
    embedded_bg_test_data,
    batch_size=32,
)

In [ ]:
autoencoder.evaluate(
    embedded_signal_data,
    embedded_signal_data,
    batch_size=32,
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(
    diff_bg_test,
    bins=50,
    alpha=0.5,
    label="Background Test Data",
    color="blue",
    density=True,
)
ax.hist(diff_signal, bins=50, alpha=0.5, label="Signal Data", color="red", density=True)
ax.hist(
    diff_bg_train,
    bins=50,
    alpha=0.5,
    label="Background Train Data",
    color="green",
    density=True,
)

ax.set_xlabel("Reconstruction Error")
ax.set_ylabel("Frequency")
ax.set_title("Reconstruction Error Distribution")
ax.legend()
plt.show()

In [ ]:
PCA = sk.decomposition.PCA(n_components=8)
test_split = int(background_data.shape[0] / 5 * 4)
train_bg_data = background_data[:test_split]
PCA.fit(train_bg_data.reshape(train_bg_data.shape[0], -1))
background_pca = generate_embeddings(train_bg_data, PCA)
test_bg_data = background_data[test_split:]
test_bg_pca = generate_embeddings(test_bg_data, PCA)

In [ ]:
signal_pca = generate_embeddings(signal_data, PCA)

In [ ]:
from src.model.components import MLP

input_layer = keras.layers.Input(shape=(pca_dim + 1,))
laten_dim = pca_dim // 5
num_layers = 4
encoder = MLP(
    laten_dim, activation="relu", name="input_embedding", num_layers=num_layers
)(input_layer)
decoder = MLP(
    pca_dim + 1, activation="linear", name="output_embedding", num_layers=num_layers
)(encoder)

autoencoder_model = keras.Model(inputs=input_layer, outputs=decoder, name="autoencoder")

In [ ]:
autoencoder_model.summary()
autoencoder_model.compile(
    optimizer=keras.optimizers.Adam(1e-2),
    loss=keras.losses.MeanSquaredError(),
    metrics=[keras.metrics.MeanSquaredError()],
)

In [ ]:
history = autoencoder_model.fit(
    background_pca,
    background_pca,
    epochs=1000,
    batch_size=256,
    validation_split=0.2,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True,
            verbose=1,
        ),
    ],
)

In [ ]:
signal_diff = signal_pca - autoencoder_model.predict(signal_pca)
signal_diff = np.linalg.norm(signal_diff, axis=1)
background_diff = test_bg_pca - autoencoder_model.predict(test_bg_pca)
background_diff = np.linalg.norm(background_diff, axis=1)

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(signal_diff, bins=50, alpha=0.5, label="Signal", color="blue", density=True)
plt.hist(
    background_diff, bins=50, alpha=0.5, label="Background", color="red", density=True
)
plt.legend()
plt.xlabel("Difference")

In [ ]:
number_hits_signal = ((signal_data != -1).any(axis=-1)).sum(axis=-1)
number_hits_background = ((test_bg_data != -1).any(axis=-1)).sum(axis=-1)

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(
    number_hits_signal, bins=50, alpha=0.5, label="Signal", color="blue", density=True
)
plt.hist(
    number_hits_background,
    bins=50,
    alpha=0.5,
    label="Background",
    color="red",
    density=True,
)
plt.legend()
plt.xlabel("Number of Hits")

In [ ]:
from sklearn.metrics import roc_curve, auc

y_true = np.concatenate([np.ones(len(signal_diff)), np.zeros(len(background_diff))])
y_scores_autoencoder = np.concatenate([signal_diff, background_diff])
y_scores_hits = np.concatenate([number_hits_signal, number_hits_background])

fpr_autoencoder, tpr_autoencoder, _ = roc_curve(y_true, y_scores_autoencoder)
fpr_hits, tpr_hits, _ = roc_curve(y_true, y_scores_hits)

roc_auc_autoencoder = auc(fpr_autoencoder, tpr_autoencoder)
roc_auc_hits = auc(fpr_hits, tpr_hits)

In [ ]:
plt.figure(figsize=(10, 10))
plt.plot(
    fpr_autoencoder,
    tpr_autoencoder,
    label=f"Autoencoder (AUC = {roc_auc_autoencoder:.2f})",
    color="blue",
)
plt.plot(
    fpr_hits, tpr_hits, label=f"Number of Hits (AUC = {roc_auc_hits:.2f})", color="red"
)
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")  # Diagonal line
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.grid()
plt.show()